## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modelbase.ode import Model, Simulator, mca
import pandas as pd
import seaborn as sns
import pickle
from model import (
    get_enzyme_costs,
    get_model,
    get_enzyme_concentrations,
    get_enzyme_parameters,
    get_proteomics_results,
    calculate_min_kcats,
)
from modelbase.ode.simulators import _Simulate
from parallel import parallelise
import itertools
from pathlib import Path
from sklearn.pipeline import Pipeline
from typing import Literal, cast
from plot import savefig
from functools import partial
from typing import TypeVar


T = TypeVar("T")


def unwrap(x: T | None) -> T:
    if x is None:
        raise ValueError("Unexpected None")
    return x


CELL_VOLUME: float = 2e-15
PROTEIN_CONTENT: float = 0.445
MODE = "blast"
CHLA = (
    7.5 / 893.509 / (105482164.18012363 * 1000)
)  # mg/OD/L -> mmol/OD/L / Cells/LLOD --> mmol/cell
CAROT = 2 / 581.5565 / (105482164.18012363 * 1000)
SQUALENE = 0.5 / 410.73 / (105482164.18012363 * 1000)
DOUBLING_TIME: float = 300.0

# Load data
DATA = Path("data")

with open(DATA / "rate_regression_model_biomass.pickle", "rb") as f:
    BIOMASS_REGRESSION_MODEL = pickle.load(f)

OCP_ABSORPTION = pd.read_csv(DATA / "OCP_absorption.csv", index_col=0)
PS_COMP = pd.read_csv(DATA / "photosystem_composition.csv", index_col=0)
MOLAR_MASSES = pd.read_csv(DATA / "molar_masses.csv", index_col=0).squeeze(1)

PIGMENTS_FUENTE = cast(
    pd.Series,
    pd.read_csv(
        DATA / "pigment_concentrations.csv",
        index_col=0,
    ).squeeze(1),
)

ABS_COEF = pd.concat(
    {
        pig: pd.read_csv(DATA / f"per_pigment/{pig}.csv", index_col=0)
        for pig in PIGMENTS_FUENTE.index
    },
    axis=1,
)

ENZYME_COSTS = get_enzyme_costs(
    kugler_model_csv=DATA / "Amino_acid_costs_Kuglermodel.csv",
    aa_costs_csv=DATA / "Amino_acid_costs_arnold.csv",
)

ABUNDANCES, DETECTION_LIMIT = get_proteomics_results(
    proteomics_diff_csv=DATA / "Proteomics_DiffAnalysis_2WayAnova_.csv",
    proteomics_poi_csv=DATA / "Proteomics_Proteins_of_interest.csv",
    proteomics_concentration_csv=DATA / "Proteomics_samples_proteinconcentration.csv",
    proteomics_dry_weight_csv=DATA / "Proteomics_Dry_weights.csv",
    proteomics_dilutions_csv=DATA / "Proteomics_samples_dilutions.csv",
    proteomics_od_wt_csv=DATA / "20240328_Spectrum_OD_WT_proteomics.csv",
    proteomics_od_sqs_csv=DATA / "20240329_Spectrum_OD_sqs_proteomics.csv",
    proteomics_od_dcas9_csv=DATA / "20240329_Spectrum_OD_dCas9_proteomics.csv",
    cell_volume=CELL_VOLUME,
    protein_content=PROTEIN_CONTENT,
)

ENZYME_CONCENTRATIONS_UMOL: dict[str, float] = get_enzyme_concentrations(
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
    abundances=ABUNDANCES,
    detection_limit=DETECTION_LIMIT,
)

ENZYME_PARAMETERS: dict[str, float] = get_enzyme_parameters(
    mode=MODE,
    blasted_parameters_csv=DATA / "enzyme_parameters_blasted.csv",
    predicted_kms_csv=DATA / "predicted_Kms.csv",
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
)

MIN_KCAT_VALUES: dict[str, float] = calculate_min_kcats(
    enzyme_concentrations_umol=ENZYME_CONCENTRATIONS_UMOL,
    cell_volume=CELL_VOLUME,
    doubling_time=DOUBLING_TIME,
)


LIGHTS = pd.DataFrame(
    {
        path.stem: pd.read_csv(path, index_col=0).squeeze(1)
        for path in sorted((DATA / "lights").glob("*.csv"))
    }
)
EXP_GROWTH_RATE_BY_DENSITY = pd.read_csv(
    DATA / "growth_rates_by_density.csv",
    index_col=0,
)["growth_rate"]

PROTEINS_OF_INTEREST = pd.read_csv(
    DATA / "Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

In [ ]:
def get_parameterised_model(
    strain: Literal["wt", "dshc"],
    growth_mode: Literal["steady", "dynamic"],
    kcat_values: dict[str, float] = MIN_KCAT_VALUES,
    cell_volume: float = CELL_VOLUME,
    enzyme_costs: pd.DataFrame = ENZYME_COSTS,
    enzyme_concentrations_umol: dict[str, float] = ENZYME_CONCENTRATIONS_UMOL,
    enzyme_parameters: dict[str, float] = ENZYME_PARAMETERS,
    biomass_regression_model: Pipeline = BIOMASS_REGRESSION_MODEL,
    abs_coef: pd.DataFrame = ABS_COEF,
    lights: pd.DataFrame = LIGHTS,
    ocp_absorption: pd.DataFrame = OCP_ABSORPTION,
    molar_masses: pd.DataFrame = MOLAR_MASSES,
    ps_comp: pd.DataFrame = PS_COMP,
) -> tuple[Model, dict[str, float]]:
    m = get_model(
        growth_mode=growth_mode,
        strain=strain,
        cell_volume=cell_volume,
        enzyme_total_cost_mol=enzyme_costs,
        enzyme_concentrations_umol=enzyme_concentrations_umol,
        enzyme_parameters=enzyme_parameters,
        biomass_regression_model=biomass_regression_model,
        abs_coef=abs_coef,
        lights=lights,
        ocp_absorption=ocp_absorption,
        molar_masses=molar_masses,
        ps_comp=ps_comp,
    )
    m.update_parameters(
        {
            # "ShcKcat": 0,  # default 600
            # "Shc_ksynth": 0, default 0.008034116888921783
            "light": 100,
            "ATP_out_metabolism": 0.5,
            "3PGA_out_metabolism": 0.5,
            "NADPH_out_metabolism": 0.5,
            "Fdred_out_metabolism": 0.5,
            "cell_volume": cell_volume,
            "DXS_inh_str": 0.001,
            "ThiGDXPKm": 0.01,
        }
    )
    m.update_parameters(
        {k: v for k, v in kcat_values.items() if k not in {"IspHKcat", "CrtEKcat"}}
    )

    # Update initial
    starting_conc = {}
    for i in m.compounds:
        starting_conc[i] = 0.01
    starting_conc.update(enzyme_concentrations_umol)
    starting_conc.update(
        {
            "3PGA": 0.1,
            "ATP": 0.1,
            "NADPH": 0.1,
            "Fdred": 0.1,
            "ThiDP": 10,
            "chlorophyll-phytol": CHLA / cell_volume,
            "Cell_density": 1e1,
            "carotenoids": CAROT / cell_volume,
            "SQ": SQUALENE / cell_volume,
        }
    )
    return m, starting_conc


def get_doubling_time(rel_growth_rate: float) -> float:
    return np.log(2) / (rel_growth_rate * 60)


# Get steady-state
ms, y0 = get_parameterised_model(
    strain="dshc",
    growth_mode="steady",
)
ms.update_parameters(
    {
        "ShcKcat": 0,  # FIXME: why is this changed?
        "Shc_ksynth": 0,  # FIXME: why is this changed?
        "light": 80,  # FIXME: why is this changed?
        "DXS_inh_str": 0,  # FIXME: why is this changed?
        "sample_depth_m": 0.0135,  # FIXME: why is this changed?
    }
)

Y0_SS = unwrap(
    Simulator(ms)
    # FIXME: explain why this number
    .initialise(y0 | {"Cell_density": EXP_GROWTH_RATE_BY_DENSITY.index[24] / 1e6})
    .simulate_to_steady_state_and()
    .get_new_y0()
)


In [ ]:
# 3 µM/min 3PGA at OD=1, growth rate 0.001 == doubling time around 11 h.
Start_OD = 1
Growth_rate = 0.001  # min^-1
print(np.log(2) / 0.001 / 60)
print(
    "g/L/min calculated from growth: ", Start_OD * (Growth_rate) / 3 / 2
)  # OD*Growth_rate min^-1--> OD/min / g/L CDW/OD --> g/L CDW/min * gC/gCDW --> g/L/min
print(
    "g/L/min calculated from influx prediction:",
    14 * 2e-15 * 1e11 * 186.06 * (36 / 186.06) * 1e-3,
)  # mmol/L/min 3PGA * L/cell--> mmol/cell/min *cells/L --> mmol/L/min * mg/mmol -> mg 3PGA /L/min --> mg C/L/min --> g/L/min
# 0.06 fg/cell s^-1
print(
    "fg/cell/s to compare to paper:", 2.4 * 2e-15 * 186.061 * 1e9 / 60
)  # mmol/L/min --> mmol/cell/min --> mg/cell/min --> fg/cell/min -_> fg/cell/s

print(
    "g/L/min calculated from fg/cell/s value in Paper:", 0.06 * 1e11 * 60 / 1e15
)  # fg/cell/s * cells/L --> fg/L/s --> fg/L/min --> g/L/min

## Fig XX: compare min kcat with literature

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.set_axisbelow(b=True)

    kcat_comp = pd.DataFrame.from_dict(MIN_KCAT_VALUES, "index")
    for key, value in ENZYME_PARAMETERS.items():
        if key in kcat_comp.index:
            kcat_comp.at[key, "A"] = value
    kcat_comp.plot(kind="bar", ax=ax)

    ax.legend([r"Calculated min $\mathrm{k_{cat}}$", r"Literature $\mathrm{k_{cat}}$"])

## Fig 09: growth rate over time

In [ ]:
# Run dynamic simulation using previous steady-state as initial conditions
md, _ = get_parameterised_model(
    strain="dshc",
    growth_mode="dynamic",
)
md.update_parameters(
    {
        "light": 80,
        "sample_depth_m": 0.0135,  # FIXME: why is this changed?
    }
)

res = unwrap(
    Simulator(md)
    # FIXME: explain why this number
    .initialise(Y0_SS | {"Cell_density": 54562873.0217531 / 1e6})
    .simulate_and(96 * 60)
    .get_full_results_df()
)

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
    }
):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")

    ax1.plot(res["Growth_rate"] * 60)
    ax1.scatter(
        x=np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY)),
        y=EXP_GROWTH_RATE_BY_DENSITY,
        c="C1",
    )
    ax1.set_ylabel("Growth rate $[h^{-1}]$")
    ax1.set_xlabel("Time [h]")
    ax1.set_xticks(np.arange(0, 97 * 60, 24 * 60))
    ax1.set_xticklabels(np.arange(0, 97, 24))
    ax1.grid()

    ax2.plot(
        res["Cell_density"] * 1e6,
        label="Simulated values",
    )
    ax2.plot(
        np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY[::2])),
        EXP_GROWTH_RATE_BY_DENSITY[::2].index,
        label="Experimental values",
    )
    ax2.set_yscale("log")
    ax2.set_ylabel(r"Cell density $[Cells\ mL^{-1}]$")
    ax2.set_xlabel("Time [h]")
    ax2.set_xticks(np.arange(0, 97 * 60, 24 * 60))
    ax2.set_xticklabels(np.arange(0, 97, 24))
    ax2.grid()

    ax2.legend(loc="lower right")
    savefig(fig, "fig9.png")
    plt.show()

## Fig 10: scan ss growth rate by cell density

In [ ]:
def _worker(
    density: float,
    model: Model,
    y0: dict[str, float],
) -> float | None:
    try:
        df = (
            Simulator(model)
            .initialise(y0 | {"Cell_density": density / 1e6})
            .simulate_to_steady_state_and(
                # simulation_kwargs={"atol": 1e-8, "rtol": 1e-8}
            )
            .get_full_results_df()
        )
        if df is not None:
            return df.iloc[-1]["Growth_rate"] * 60
    except:  # noqa: E722
        print(f"Failed for density={density}")
    return None


ms, y0 = get_parameterised_model(
    strain="dshc",
    growth_mode="steady",
)
ms.update_parameters(
    {
        "light": 80,  # FIXME: why is this changed?
        "sample_depth_m": 0.02,  # FIXME: why is this changed?
    }
)

growth_rate_by_density = pd.Series(
    parallelise(
        partial(_worker, model=ms, y0=y0),
        inputs=list(
            zip(
                EXP_GROWTH_RATE_BY_DENSITY.index[::4],  # FIXME: 2
                EXP_GROWTH_RATE_BY_DENSITY.index[::4],  # FIXME: 2
                strict=True,
            )
        ),
    ),
    dtype=float,
)


In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, ax = plt.subplots(figsize=(4, 3))
    growth_rate_by_density.plot(ax=ax, label="Simulation")
    EXP_GROWTH_RATE_BY_DENSITY.plot(ax=ax, marker="o", label="Experimental")
    ax.set_xscale("log")
    ax.set_ylim(0)
    ax.set_ylabel(r"Growth rate [$h^{-1}$]")
    ax.set_xlabel(r"Cell density [$Cells\ ml^{-1}$]")
    ax.legend()
    savefig(fig, "fig10.png")
    plt.show()

## Growth repression induction time point tests

Unmodified dshc strain

In [ ]:
def repress_crte(s: _Simulate) -> _Simulate:
    s.update_parameter("CrtE3Kcat", 0.01)
    return s


def product_after_induction(
    time_point_of_induction: float,
    y0_ss: dict[str, float],
) -> float:
    md, y0 = get_parameterised_model(
        strain="dshc",
        growth_mode="dynamic",
    )

    s = Simulator(md).initialise(y0_ss).simulate_and(time_point_of_induction)
    s = repress_crte(s)
    s.simulate(fourteen_days_in_min)

    if (c := s.get_full_results_df()) is None:
        return 0

    return c.iloc[-1]


day_in_min = 60 * 24
fourteen_days_in_min = 14 * day_in_min
time_points_of_induction = np.arange(
    0,
    fourteen_days_in_min,
    day_in_min,
    dtype=float,
)

res = pd.DataFrame(
    parallelise(
        partial(product_after_induction, y0_ss=Y0_SS),
        list(zip(time_points_of_induction, time_points_of_induction, strict=True)),
    )
).T

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, (ax1, ax2, ax3) = plt.subplots(
        1,
        3,
        figsize=(8, 3),
        layout="constrained",
    )

    fig.suptitle("Total concentration after 14 days")

    ax1.plot(
        res.index / day_in_min,
        res["Cell_density"] * 1e9 * res["SQ"] * 410.73 * CELL_VOLUME,
    )
    ax1.set_ylabel(
        "Squalene conc. in culture\n"
        r"[$mg\ L^{-1}$]"
    )

    ax2.plot(
        res.index / day_in_min,
        res["Cell_density"] * 1e9,
    )
    ax2.set_yscale("log")
    ax2.set_ylabel(
        "Cell density\n"
        r"[$Cells\ L^{-1}$]"
    )

    ax3.plot(
        res.index / day_in_min,
        res["SQ"] * 410.73 / 1000,
    )
    ax3.set_ylabel(
        "Intracellular squalene\n"
        r"concentration [$g\ L^{-1}$]"
    )

    for ax in (ax1, ax2, ax3):
        ax.set_xlabel("Timepoint of induction [d]")

    plt.show()


## Quadruple overexpression of Sqs, DXS, IspD, IspG

In [ ]:
def product_after_induction(
    time_point_of_induction: float,
    y0_ss: dict[str, float],
    poi: pd.DataFrame,
) -> float:
    md, y0 = get_parameterised_model(
        strain="dshc",
        growth_mode="dynamic",
    )

    for gene in ["Sqs", "IspD", "IspG", "DXS"]:
        overexpression_amount = 1.331057 * poi[poi["Enzyme"] == gene]["kDa"]
        md.update_parameters(
            {
                gene + "_ksynth": md.get_parameter(gene + "_ksynth")
                + overexpression_amount / (5.5 * 60)
            }
        )

    s = Simulator(md).initialise(y0_ss).simulate_and(time_point_of_induction)
    s = repress_crte(s)
    s.simulate(fourteen_days_in_min)

    if (c := s.get_full_results_df()) is None:
        return 0

    return c.iloc[-1]


day_in_min = 60 * 24
fourteen_days_in_min = 14 * day_in_min
time_points_of_induction = np.arange(0, fourteen_days_in_min, day_in_min, dtype=float)
res_ox_strain = pd.DataFrame(
    parallelise(
        partial(product_after_induction, y0_ss=Y0_SS, poi=PROTEINS_OF_INTEREST),
        list(zip(time_points_of_induction, time_points_of_induction, strict=True)),
    )
).T

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 20))
for r in res_ox_strain:
    ax[0].scatter(
        x=r,
        y=res_ox_strain[r]["Cell_density"]
        * 1e9
        * res_ox_strain[r]["SQ"]
        * 410.73
        * CELL_VOLUME,
        c="blue",
    )  # Conversion from intracellular mM to mg/L squalene
    ax[1].scatter(x=r, y=res_ox_strain[r]["Cell_density"] * 1e9, c="green")
    ax[2].scatter(x=r, y=res_ox_strain[r]["SQ"] * 410.73 / 1000, c="blue")
    # plt.scatter(x=r, y=res_ox_strain[r]["Cell_density"]*)
ax[0].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[0].set_xticklabels(np.arange(0, 15, 1))
ax[1].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[1].set_xticklabels(np.arange(0, 15, 1))
ax[2].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[2].set_xticklabels(np.arange(0, 15, 1))
ax[0].set_ylabel("Squalene concentration in culture after 14 d [$mg\ L^{-1}$]")

ax[1].set_ylabel("Cell density after 14 d [$Cells\ L^{-1}$]")
ax[2].set_ylabel("Intracellular squalene concentration after 14 d [$g\ L^{-1}$]")
ax[0].set_xlabel("Timepoint of induction [d]")
ax[1].set_xlabel("Timepoint of induction [d]")
ax[2].set_xlabel("Timepoint of induction [d]")

In [ ]:
ms = get_parameterised_model()

s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
Y0_SS = dict(zip(ms.compounds, y_ss[-1]))
Y0_SS

In [ ]:
get_doubling_time(s.get_full_results_df()["Growth_rate"])

## Simulation of overexpressions in different combinations, maybe include different strengths?

In [ ]:
def_model = get_parameterised_model()

s = Simulator(def_model)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
Y0_SS = dict(zip(def_model.compounds, y_ss[-1]))

CELL_VOLUME = 2e-15
MIN_KCAT_VALUES = calculate_min_kcats(cell_volume=CELL_VOLUME, doubling_time=300)
ms = get_model(mode="blast", growth_mode="dynamic", strain="dshc")
ms.update_parameters(
    {
        "ShcKcat": 0,
        "Shc_ksynth": 0,
        "light": 100,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,  # .0003137085008420509,
        # "ATP_tot":430.343, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
        "cell_volume": CELL_VOLUME,
        "protein_content": 0.445,
        "DXS_inh_str": 0.001,
    }
)
ms.update_parameters(MIN_KCAT_VALUES)
poi = pd.read_csv(
    "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

c_flux = []
conc_collection_ox = []
flux_collection_ox = []

genes = ["Sqs", "IspG", "DXS", "IspD", "Idi"]

# Generate all possible combinations of the genes
for r in range(1, len(genes) + 1):
    for combination in itertools.combinations(genes, r):
        ms = get_model(mode="blast", growth_mode="steady", strain="dshc")
        for gene in combination:
            overexpression_amount = 1.331057 * poi[poi["Enzyme"] == gene]["kDa"]
            ms.update_parameters(
                {
                    gene + "_ksynth": def_model.get_parameter(gene + "_ksynth")
                    + overexpression_amount
                }
            )
        ms.update_parameters(
            {
                "light": 100,
                "ATP_out_metabolism": 0.5,
                "3PGA_out_metabolism": 0.5,
                "NADPH_out_metabolism": 0.5,
                "Fdred_out_metabolism": 0.5,
                "out": 0,  # .0003137085008420509,
                # "ATP_tot":430.143, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
                "cell_volume": CELL_VOLUME,
                "protein_content": 0.445,
                "DXS_inh_str": 0.001,
            }
        )
        ms.update_parameters(MIN_KCAT_VALUES)
        s = Simulator(ms)
        y_ss_modified = dict((zip(ms.compounds, y_ss[-1])))
        y_ss_modified.update({"Cell_density": 1e2})
        s.initialise(Y0_SS)
        t_steady, y_ss = s.simulate_to_steady_state()
        conc_collection_ox.append(s.get_full_results_df())
        flux_collection_ox.append(s.get_fluxes_df())


In [ ]:
# Assuming conc_collection is a list of DataFrames
# and each DataFrame has a column named "SQ"
fig, ax = plt.subplots(figsize=(10, 10))
# Extract the "SQ" values from each DataFrame in conc_collection
print([df for df in flux_collection_ox])

sq_values = [
    df["Sqs"].iloc[-1] / df["Biomass"].iloc[-1] * 410.73 * 1e3 * 1e9 * CELL_VOLUME
    for df in flux_collection_ox
]

# Generate the labels for the legend based on the gene combinations
genes = ["Sqs", "IspG", "DXS", "IspD", "Idi"]
labels = []
for r in range(1, len(genes) + 1):
    for combination in itertools.combinations(genes, r):
        labels.append(", ".join(combination))

# Create a bar plot
x = np.arange(len(sq_values))
plt.bar(x, sq_values)

# Add labels and legend
ax.set_xlabel("Gene Combinations")
ax.set_ylabel("SQ Values")
ax.set_title("SQ Values for Different Gene Combinations")
ax.set_xticks(x, labels, rotation=45, ha="right")
ax.legend(labels, bbox_to_anchor=(1.05, 1), loc="upper left")

# Show the plot
plt.tight_layout()
plt.show()


## Gradient descent MCA, with no repeat overexpression allowed

In [ ]:
poi = pd.read_csv(
    "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

ms = get_parameterised_model()
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
y0 = {}
for i in ms.compounds:
    y0[i] = 0.01
y0.update(get_enzyme_concentrations())
y0.update(
    {
        "3PGA": 0.1,
        "ATP": 0.1,
        "NADPH": 0.1,
        "Fdred": 0.1,
        "ThiDP": 10,
        "chlorophyll-phytol": CHLA / ms.get_parameter("cell_volume"),
        "Cell_density": 1e1,
        "carotenoids": CAROT / ms.get_parameter("cell_volume"),
        "SQ": SQUALENE / ms.get_parameter("cell_volume"),
    }
)
s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()

Y0_SS = y_ss[-1]
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
frcs_one_ox = {}
mca_results_one_ox = {}
modified_parameters_one_ox = []  # To keep track of modified parameters in order

for runs in np.arange(0, 10):
    crc, frc = mca.get_response_coefficients_df(ms, pars_for_mca, Y0_SS)
    s = Simulator(ms)
    s.initialise(Y0_SS)
    t_ss, y_ss = s.simulate_to_steady_state()
    mca_results_one_ox[runs] = s.get_fluxes_df().iloc[-1]
    frcs_one_ox[runs] = frc.copy()
    Y0_SS = y_ss[-1]

    # Filter out already modified parameters
    # frc_filtered = frc.copy()
    frc_filtered = frc.drop(index=modified_parameters_one_ox)
    most_positive = (frc_filtered["Sqs"] + frc_filtered["Biomass"]).idxmax()
    overexpression_amount = 1.331057 * poi[poi["Enzyme"] == most_positive[:-7]]["kDa"]
    ms.update_parameters(
        {
            most_positive: ms.get_parameter(most_positive)
            + overexpression_amount / (5.5 * 60)
        }
    )
    modified_parameters_one_ox.append(
        most_positive
    )  # Add the modified parameter to the list

# Now, modified_parameters contains the parameters in the order they were modified


In [ ]:
fig, ax = plt.subplots(2, figsize=(10, 10))
# Assuming frcs_one_ox and modified_parameters are defined as in your previous code
s_coll = pd.DataFrame()
Squalene_coll = []
for i, f in enumerate(frcs_one_ox):
    s_coll = pd.concat(
        [
            s_coll,
            (frcs_one_ox[f]["Sqs"] + abs(frcs_one_ox[f]["Biomass"].min())).rename(f),
        ],
        axis=1,
    )
    # s_coll = pd.concat([s_coll, (frcs_one_ox[f]["Biomass"]).rename(f)], axis=1)

    Squalene_coll.append(
        mca_results_one_ox[i]["Sqs"] / mca_results_one_ox[i]["Biomass"] * 410.73 / 1000
    )

print(s_coll)

# Create the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(s_coll, annot=False, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[0])

ax[1].plot(Squalene_coll)
ax[1].set_ylim(0)
# Draw rectangles around the overexpressed genes
for run, gene in enumerate(modified_parameters_one_ox):
    ax[0].add_patch(
        plt.Rectangle(
            (run, s_coll.index.get_loc(gene)),
            1,
            1,
            fill=False,
            edgecolor="red",
            linewidth=3,
        )
    )
ax[0].set_xlabel("No. of genes overexpressed")
ax[0].set_ylabel("Adjustable parameters")
ax[1].set_ylabel("Intracellular squalene concentration [$g\ L^{-1}$]")
ax[1].set_xlabel("No. of genes overexpressed")
ax[1].set_xticks(np.arange(0, 10, 1))
plt.show()

# SQ_coll.append(mca_results_one_ox[i]["Biomass"]*mca_results_one_ox[i]["Sqs"])